# HFA AI Training - Module 3: Vector Databases
## Understanding Embeddings

This notebook demonstrates how embeddings work:
1. Generating embeddings for words and sentences
2. Measuring similarity between embeddings (cosine similarity)
3. Showing that semantically similar things cluster together
4. The classic "king - man + woman ≈ queen" analogy

This example uses sentence-transformers, which runs locally and is completely free (no API key needed).

Optionally, you can use OpenAI embeddings by setting:
```bash
export OPENAI_API_KEY="sk-your-key-here"
export USE_OPENAI_EMBEDDINGS="true"
```

### Install Dependencies

In [ ]:
!pip install sentence-transformers numpy scikit-learn

### Section 1: Choose an Embedding Model

We support two approaches:
- **sentence-transformers** (free, local, no API key)
- **OpenAI embeddings** (paid, requires API key)

We default to sentence-transformers for ease of use.

In [ ]:
import os
import numpy as np

# ============================================================
# SECTION 1: Choose an Embedding Model
# ============================================================
# We support two approaches:
#   - sentence-transformers (free, local, no API key)
#   - OpenAI embeddings (paid, requires API key)
#
# We default to sentence-transformers for ease of use.

USE_OPENAI = os.environ.get("OPENAI_API_KEY") is not None and os.environ.get("USE_OPENAI_EMBEDDINGS", "").lower() == "true"

if USE_OPENAI:
    # ------- OpenAI Embedding Approach -------
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    MODEL_NAME = "text-embedding-3-small"

    def get_embedding(text: str) -> np.ndarray:
        """Get an embedding vector from OpenAI's API."""
        response = client.embeddings.create(input=text, model=MODEL_NAME)
        return np.array(response.data[0].embedding)

    def get_embeddings_batch(texts: list[str]) -> list[np.ndarray]:
        """Get embeddings for multiple texts in one API call (more efficient)."""
        response = client.embeddings.create(input=texts, model=MODEL_NAME)
        return [np.array(item.embedding) for item in response.data]

    print(f"Using OpenAI embeddings ({MODEL_NAME})")
    print("=" * 60)

else:
    # ------- Local Sentence-Transformers Approach -------
    # This model runs entirely on your machine — no API key needed.
    # 'all-MiniLM-L6-v2' is a small, fast model that produces
    # 384-dimensional embeddings. Good for learning and demos.
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")

    def get_embedding(text: str) -> np.ndarray:
        """Get an embedding vector from the local model."""
        return model.encode(text)

    def get_embeddings_batch(texts: list[str]) -> list[np.ndarray]:
        """Get embeddings for multiple texts at once."""
        return [model.encode(t) for t in texts]

    print("Using local sentence-transformers (all-MiniLM-L6-v2)")
    print("=" * 60)

### Section 2: Cosine Similarity Function

Cosine similarity measures the angle between two vectors. It ranges from -1 (opposite) to 1 (identical). We use it to determine how "similar" two embeddings are.

In [ ]:
# ============================================================
# SECTION 2: Cosine Similarity Function
# ============================================================
# Cosine similarity measures the angle between two vectors.
# It ranges from -1 (opposite) to 1 (identical).
# We use it to determine how "similar" two embeddings are.

def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """
    Calculate the cosine similarity between two vectors.

    Formula: cos(theta) = (A . B) / (|A| * |B|)

    Where:
      A . B   = dot product of vectors A and B
      |A|     = magnitude (length) of vector A
      |B|     = magnitude (length) of vector B

    Returns a float between -1 and 1:
      1.0  = identical direction (same meaning)
      0.0  = perpendicular (unrelated)
     -1.0  = opposite direction (opposite meaning)
    """
    dot_product = np.dot(vec_a, vec_b)
    magnitude_a = np.linalg.norm(vec_a)
    magnitude_b = np.linalg.norm(vec_b)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    return dot_product / (magnitude_a * magnitude_b)

### Section 3: Basic Word Embeddings

Let's embed some individual words and see how the model captures meaning. Words with similar meanings should have similar embeddings.

In [ ]:
# ============================================================
# SECTION 3: Basic Word Embeddings
# ============================================================
# Let's embed some individual words and see how the model
# captures meaning. Words with similar meanings should have
# similar embeddings.

print("\n--- PART 1: Word Embeddings ---\n")

# A set of words from different categories
words = [
    # Real estate terms
    "house", "home", "property", "residence", "dwelling",
    # Nature/location terms
    "beach", "ocean", "mountain", "forest",
    # Unrelated terms
    "computer", "algorithm", "software",
]

# Generate embeddings for all words
print(f"Generating embeddings for {len(words)} words...")
word_embeddings = {word: get_embedding(word) for word in words}

# Show the shape of an embedding vector
sample_word = "house"
sample_vec = word_embeddings[sample_word]
print(f"\nThe word '{sample_word}' is represented as a vector of {len(sample_vec)} numbers.")
print(f"First 10 values: [{', '.join(f'{v:.4f}' for v in sample_vec[:10])}, ...]")

### Word Similarity Comparisons

Now let's compare pairs of words to see how similar their embeddings are. Synonyms should have high similarity, while unrelated words should have low similarity.

In [ ]:
# Compare some word pairs
print("\n--- Word Similarity Comparisons ---\n")
pairs_to_compare = [
    ("house", "home"),        # Very similar — synonyms
    ("house", "property"),    # Similar — related concept
    ("house", "residence"),   # Similar — formal synonym
    ("house", "beach"),       # Somewhat related (beach house?)
    ("house", "computer"),    # Unrelated
    ("beach", "ocean"),       # Related — same domain
    ("ocean", "mountain"),    # Both nature, but different
    ("computer", "software"), # Related — tech domain
]

for word_a, word_b in pairs_to_compare:
    sim = cosine_similarity(word_embeddings[word_a], word_embeddings[word_b])
    # Create a visual bar showing similarity
    bar_length = int(sim * 30)
    bar = "#" * max(bar_length, 0) + "." * (30 - max(bar_length, 0))
    print(f"  {word_a:>12} <-> {word_b:<12}  sim: {sim:.4f}  [{bar}]")

### Section 4: Sentence Embeddings

Now let's embed full sentences. This is more powerful because it captures the meaning of entire phrases, not just individual words.

In [ ]:
# ============================================================
# SECTION 4: Sentence Embeddings
# ============================================================
# Now let's embed full sentences. This is more powerful because
# it captures the meaning of entire phrases, not just individual words.

print("\n\n--- PART 2: Sentence Embeddings ---\n")

# Real estate-themed sentences with varying similarity
sentences = [
    "Beautiful 3-bedroom home with ocean views in Kailua",
    "Stunning three bed house overlooking the sea in Kailua",  # Same meaning, different words!
    "Affordable starter home in Ewa Beach",
    "Luxury penthouse in downtown Honolulu",
    "The best recipe for chocolate chip cookies",  # Completely unrelated
]

print("Embedding sentences and comparing similarity:\n")

# Embed all sentences
sentence_embeddings = {s: get_embedding(s) for s in sentences}

# Compare the first sentence to all others
reference = sentences[0]
print(f"  Reference: \"{reference}\"\n")

for i, sentence in enumerate(sentences[1:], 1):
    sim = cosine_similarity(sentence_embeddings[reference], sentence_embeddings[sentence])
    bar_length = int(sim * 30)
    bar = "#" * max(bar_length, 0) + "." * (30 - max(bar_length, 0))
    print(f"  [{sim:.4f}] [{bar}]")
    print(f"    \"{sentence}\"\n")

print("  Notice how the first two sentences have very high similarity")
print("  even though they use completely different words!")
print("  The cookie recipe has very low similarity (different topic entirely).")

### Section 5: Vector Arithmetic (Analogies)

This demonstrates that embeddings capture relationships. If we take the "king" vector, subtract the "man" direction, and add the "woman" direction, we should end up near "queen."

**NOTE:** This works best with models trained on large corpora. Results with small models may vary, but the concept holds.

In [ ]:
# ============================================================
# SECTION 5: The Classic Analogy — "King - Man + Woman ≈ Queen"
# ============================================================
# This demonstrates that embeddings capture relationships.
# If we take the "king" vector, subtract the "man" direction,
# and add the "woman" direction, we should end up near "queen."
#
# NOTE: This works best with models trained on large corpora.
# Results with small models may vary, but the concept holds.

print("\n\n--- PART 3: Vector Arithmetic (Analogies) ---\n")

# Get embeddings for our analogy words
analogy_words = ["king", "man", "woman", "queen", "prince", "princess", "boy", "girl"]
analogy_embeddings = {w: get_embedding(w) for w in analogy_words}

# Perform the classic analogy: king - man + woman = ???
# This should be closest to "queen"
result_vector = (
    analogy_embeddings["king"]
    - analogy_embeddings["man"]
    + analogy_embeddings["woman"]
)

print("  Computing: king - man + woman = ???\n")
print("  Similarity of result to each word:")

# Check which word the result is closest to
similarities = []
for word in analogy_words:
    sim = cosine_similarity(result_vector, analogy_embeddings[word])
    similarities.append((word, sim))

# Sort by similarity (highest first)
similarities.sort(key=lambda x: x[1], reverse=True)

for word, sim in similarities:
    marker = " <-- CLOSEST!" if word == similarities[0][0] else ""
    bar_length = int(sim * 30)
    bar = "#" * max(bar_length, 0) + "." * (30 - max(bar_length, 0))
    print(f"    {word:>10}: {sim:.4f}  [{bar}]{marker}")

print(f"\n  The result of 'king - man + woman' is closest to: '{similarities[0][0]}'")

### Real Estate Analogy

Let's try a real estate-themed analogy: **mansion - expensive + affordable = ???**

In [ ]:
# Let's try another analogy relevant to real estate
print("\n\n  --- Real Estate Analogy ---")
print("  Computing: mansion - expensive + affordable = ???\n")

re_words = ["mansion", "expensive", "affordable", "cottage", "house",
            "villa", "cabin", "apartment", "shack", "bungalow"]
re_embeddings = {w: get_embedding(w) for w in re_words}

result_vector_re = (
    re_embeddings["mansion"]
    - re_embeddings["expensive"]
    + re_embeddings["affordable"]
)

re_similarities = []
for word in re_words:
    if word not in ("mansion", "expensive", "affordable"):
        sim = cosine_similarity(result_vector_re, re_embeddings[word])
        re_similarities.append((word, sim))

re_similarities.sort(key=lambda x: x[1], reverse=True)

print("  Similarity of result to candidate words:")
for word, sim in re_similarities:
    bar_length = int(sim * 30)
    bar = "#" * max(bar_length, 0) + "." * (30 - max(bar_length, 0))
    print(f"    {word:>12}: {sim:.4f}  [{bar}]")

### Section 6: Visualizing Embedding Clusters

Let's embed a larger set of terms and see how they cluster by category. Words from the same category (Real Estate, Beach/Ocean, Technology) should be more similar to each other than to words from different categories.

In [ ]:
# ============================================================
# SECTION 6: Visualizing Embedding Clusters
# ============================================================
# Let's embed a larger set of terms and see how they cluster
# by category. We'll print a simple text-based analysis.

print("\n\n--- PART 4: Embedding Clusters ---\n")

# Words grouped by category (but we'll embed them all together)
categories = {
    "Real Estate": ["house", "condo", "apartment", "mansion", "property", "listing"],
    "Beach/Ocean": ["beach", "ocean", "surf", "waves", "shoreline", "coastal"],
    "Technology":  ["computer", "software", "algorithm", "database", "code", "programming"],
}

# Flatten and embed
all_terms = []
term_to_category = {}
for category, terms in categories.items():
    for term in terms:
        all_terms.append(term)
        term_to_category[term] = category

all_embeds = {term: get_embedding(term) for term in all_terms}

# For each term, find its most similar terms and show category coherence
print("  For each word, showing its top-3 most similar words:")
print("  (Words from the same category should cluster together)\n")

for term in all_terms[:6]:  # Show first 6 to keep output manageable
    sims = []
    for other in all_terms:
        if other != term:
            sim = cosine_similarity(all_embeds[term], all_embeds[other])
            sims.append((other, sim, term_to_category[other]))
    sims.sort(key=lambda x: x[1], reverse=True)

    category = term_to_category[term]
    print(f"  '{term}' [{category}]:")
    for other, sim, other_cat in sims[:3]:
        match = "SAME" if other_cat == category else "DIFF"
        print(f"    -> '{other}' [{other_cat}]  sim: {sim:.4f}  ({match} category)")
    print()

### Key Takeaways

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("=" * 60)
print("KEY TAKEAWAYS:")
print("=" * 60)
print("""
  1. Embeddings convert text into numerical vectors (lists of numbers).
  2. Similar meanings produce similar vectors (high cosine similarity).
  3. This works for words, sentences, and even images.
  4. Vector arithmetic captures relationships (king - man + woman = queen).
  5. Words naturally cluster by topic in embedding space.
  6. This is the foundation for semantic search and vector databases!
""")